# 节点 07 — 手撕 Bahdanau Attention（注意力机制）

**目标**：
1. 手写一个最小版本的 Bahdanau attention，零依赖（只用 Python + NumPy）
2. 验证注意力权重之和严格等于 1（softmax 的数学保证）
3. 可视化：看到注意力如何在输入序列上分配

**前置知识**：会 Python 列表/循环，懂 `sum()` 是什么即可。

## 第一步：回顾 softmax——把任意分数变成概率

给定一组分数 $e_1, e_2, \ldots, e_T$，softmax 把它们变成权重：

$$\alpha_j = \frac{\exp(e_j)}{\sum_{k=1}^T \exp(e_k)}$$

**数学保证**：
- 每个 $\alpha_j \geq 0$（因为 $\exp$ 永远为正）
- 所有 $\alpha_j$ 之和严格等于 1（分子之和 = 分母）

In [ ]:
import numpy as np

def softmax(scores):
    """把一组分数转为概率分布（权重之和=1）。"""
    # 减最大值防止 exp 溢出（数学等价，但数值更稳定）
    scores = np.array(scores, dtype=float)
    exps = np.exp(scores - np.max(scores))
    return exps / exps.sum()

# 验证 softmax 的数学性质
test_scores = [2.0, 1.0, 0.5, -1.0]
weights = softmax(test_scores)

print('输入分数:', test_scores)
print('Softmax 输出:', np.round(weights, 4))
print(f'权重之和: {weights.sum():.10f}')
print(f'所有权重 >= 0: {all(weights >= 0)}')

# 核心断言：权重之和必须等于 1（softmax 的数学保证）
assert abs(weights.sum() - 1.0) < 1e-9, f'softmax 权重之和 = {weights.sum()}, 应为 1.0'
print('断言通过：权重之和 = 1（到小数点后9位）')

## 第二步：编码器——每个输入词得到一个向量

在真实 seq2seq 里，编码器是一个 RNN（用节点06学过的 LSTM）。
这里我们用随机向量模拟：重点是理解**形状**，而不是 RNN 细节。

输入序列（4个词）→ 编码器输出 4 个隐藏状态，每个是 8 维向量。

In [ ]:
np.random.seed(42)  # 固定随机种子，确保结果可重复

# 模拟输入："The bank was robbed"
source_words = ['The', 'bank', 'was', 'robbed']
T = len(source_words)   # 源句长度
d_h = 8                 # 隐藏层维度（真实模型用 256 或 512）

# 编码器输出：T 个隐藏状态，每个是 d_h 维
# 形状：(T, d_h) = (4, 8)
encoder_states = np.random.randn(T, d_h)  # 这里用随机数模拟

print(f'源句长度 T = {T}')
print(f'隐藏层维度 d_h = {d_h}')
print(f'编码器输出形状: {encoder_states.shape}')
print()
for i, word in enumerate(source_words):
    print(f'  h_{i+1} = encoder_states[{i}]  (对应词: {repr(word)})')

## 第三步：打分函数——计算相关度

Bahdanau 使用加性（additive）注意力打分：

$$e_j = \mathbf{v}^\top \tanh(\mathbf{W}_1 \, h_j + \mathbf{W}_2 \, s)$$

其中：
- $h_j$：编码器状态（8维）
- $s$：解码器当前状态（8维）
- $\mathbf{W}_1, \mathbf{W}_2$：权重矩阵（形状 8×8），可学习
- $\mathbf{v}$：权重向量（8维），可学习
- $e_j$：标量分数，越大代表第 j 个输入和当前解码状态越相关

这里我们用随机初始化的权重演示机制——真实训练会让这些参数学会相关度的含义。

In [ ]:
# 可学习参数（真实训练中由反向传播更新）
d_a = 8  # 注意力中间层维度
W1 = np.random.randn(d_a, d_h) * 0.1  # (8, 8)
W2 = np.random.randn(d_a, d_h) * 0.1  # (8, 8)
v  = np.random.randn(d_a) * 0.1       # (8,)

# 解码器当前状态
decoder_state = np.random.randn(d_h)  # (8,)

def bahdanau_score(h_j, s, W1, W2, v):
    """计算编码器状态 h_j 和解码器状态 s 的相关度分数。"""
    # tanh(W1*h_j + W2*s) 把两个向量融合，映射到 (-1, 1) 区间
    combined = np.tanh(W1 @ h_j + W2 @ s)
    # v^T * combined：把向量压成一个标量分数
    return v @ combined

# 对每个源语言位置计算分数
scores = np.array([
    bahdanau_score(encoder_states[j], decoder_state, W1, W2, v)
    for j in range(T)
])

print('各位置的相关度分数 e_j:')
for j, (word, score) in enumerate(zip(source_words, scores)):
    print(f'  e_{j+1} = {score:.4f}  (词: {repr(word)})')

## 第四步：Softmax → 注意力权重

把分数变成权重，保证所有权重加起来 = 1。

In [ ]:
# 计算注意力权重
alphas = softmax(scores)

print('注意力权重（注意力在各输入词上的分配）:')
for j, (word, alpha) in enumerate(zip(source_words, alphas)):
    bar = chr(9608) * int(alpha * 40)  # 简单可视化
    print(f'  alpha_{j+1} = {alpha:.4f}  {bar}  {repr(word)}')

print(f'\n权重之和 = {alphas.sum():.10f}')

# 核心断言：softmax 的数学性质——权重之和必须等于 1
assert abs(alphas.sum() - 1.0) < 1e-9, \
    f'注意力权重之和 = {alphas.sum():.10f}, 应严格等于 1.0'
print('断言通过：注意力权重之和 = 1（softmax 的数学保证）')

# 额外验证：所有权重必须为正
assert all(alphas >= 0), '注意力权重必须非负'
print('断言通过：所有注意力权重 >= 0')

## 第五步：加权求和 → 上下文向量

$$c = \sum_{j=1}^{T} \alpha_j \cdot h_j$$

上下文向量 $c$ 是所有编码器状态的加权平均：
- 权重大的位置贡献多
- 权重小的位置贡献少

In [ ]:
# 上下文向量：加权求和
# alphas 形状: (T,), encoder_states 形状: (T, d_h)
# alphas[:, None] 广播到 (T, d_h) 后相乘，再按行求和
context_vector = (alphas[:, None] * encoder_states).sum(axis=0)

print(f'上下文向量 c 的形状: {context_vector.shape}')
print(f'上下文向量（前4维）: {context_vector[:4].round(4)}')

# 验证：上下文向量是编码器状态的凸组合（在其范围内）
# 即：每一维的值应在 min(h_j[dim]) 和 max(h_j[dim]) 之间
for dim in range(d_h):
    lo = encoder_states[:, dim].min()
    hi = encoder_states[:, dim].max()
    val = context_vector[dim]
    assert lo <= val <= hi, \
        f'维度 {dim}: 上下文向量 {val:.4f} 不在 [{lo:.4f}, {hi:.4f}] 内'

print('断言通过：上下文向量是编码器状态的凸组合（每维在合法范围内）')

## 第六步：完整的单步注意力函数

把前面的步骤打包成一个函数，用于解码器的每一步。

In [ ]:
def bahdanau_attention(encoder_states, decoder_state, W1, W2, v):
    """
    单步 Bahdanau 注意力。

    参数:
        encoder_states: shape (T, d_h) -- 编码器全部隐藏状态
        decoder_state:  shape (d_h,)   -- 解码器当前状态
        W1, W2:         shape (d_a, d_h) -- 可学习权重
        v:              shape (d_a,)   -- 可学习权重向量

    返回:
        context: shape (d_h,) -- 上下文向量
        alphas:  shape (T,)   -- 注意力权重（和=1）
    """
    T = len(encoder_states)

    # 步骤1：计算每个位置的相关度分数
    scores = np.array([
        v @ np.tanh(W1 @ encoder_states[j] + W2 @ decoder_state)
        for j in range(T)
    ])

    # 步骤2：softmax 归一化
    alphas = softmax(scores)

    # 步骤3：加权求和
    context = (alphas[:, None] * encoder_states).sum(axis=0)

    return context, alphas


# 测试完整函数
context, alphas = bahdanau_attention(encoder_states, decoder_state, W1, W2, v)

print('完整注意力函数测试:')
print(f'  输入序列长度 T = {T}')
print(f'  上下文向量形状 = {context.shape}')
print(f'  注意力权重形状 = {alphas.shape}')
print(f'  权重之和 = {alphas.sum():.10f}')

assert abs(alphas.sum() - 1.0) < 1e-9
assert context.shape == (d_h,)
assert alphas.shape == (T,)
print('所有断言通过')

## 第七步：多步解码——模拟生成 3 个目标词

每次生成一个词，解码器状态更新，注意力权重也会改变。

In [ ]:
# 目标词（我们假设要生成 3 个词）
target_words = ['那家', '银行', '被抢了']
n_target = len(target_words)

# 模拟解码器状态序列（真实中由 RNN 更新）
decoder_states = np.random.randn(n_target, d_h)

print('每步生成的注意力分布:\n')
header = f'{"":10s}' + ''.join(f'{w:>10s}' for w in source_words)
print(header)
print('-' * 50)

all_alphas = []
for i, (tgt_word, s_i) in enumerate(zip(target_words, decoder_states)):
    _, alphas_i = bahdanau_attention(encoder_states, s_i, W1, W2, v)
    all_alphas.append(alphas_i)
    row = f'{tgt_word:10s}' + ''.join(f'{a:>10.3f}' for a in alphas_i)
    print(row)

# 验证：每一行的权重之和 = 1
for i, alphas_i in enumerate(all_alphas):
    assert abs(alphas_i.sum() - 1.0) < 1e-9, \
        f'第 {i+1} 步注意力权重之和 = {alphas_i.sum()}'

print(f'\n所有 {n_target} 步的注意力权重之和均严格等于 1')

## 总结：注意力机制的核心思想

| 步骤 | 操作 | 含义 |
|------|------|------|
| 1 | 编码器输出 $h_1, \ldots, h_T$ | 每个位置都保存状态，不压缩 |
| 2 | 打分 $e_j = \mathbf{v}^\top \tanh(W_1 h_j + W_2 s)$ | 计算当前状态和哪个位置最相关 |
| 3 | Softmax 得到 $\alpha_j$（和=1） | 把分数变成概率分布 |
| 4 | 加权求和 $c = \sum \alpha_j h_j$ | 提取当前最需要的信息 |

**关键突破**：信息不再被压缩成单一向量——解码器每一步都能看遍整个输入。

**下一站**：2017 年，Vaswani et al. 问了一个问题：如果我们不用 RNN，只用注意力呢？
答案就是 Transformer——节点09的主角。